In [1]:
import pandas as pd
import re
import time
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
import string
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

c:\Users\Softlaptop\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Softlaptop\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
df1 = pd.read_csv("../Datasets/Final_Data.csv")
arab_df = df1.copy()
arab_df.head(10)

,review_description,rating,company
0,رائع,positive,talbat
1,برنامج رائع جدا يساعد على تلبيه الاحتياجات بشك...,positive,talbat
2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال با...,negative,talbat
3,لماذا لا يمكننا طلب من ماكدونالدز؟,negative,talbat
4,البرنامج بيظهر كل المطاعم و مغلقه مع انها بتكو...,negative,talbat
5,أصبح غالي جداً,negative,talbat
6,جميل جدا رائع. . .,positive,talbat
7,للأسف الواحد ينصدم بعد زيادة الاسعار و للاسف ب...,negative,talbat
8,برنامج توترز توصيل احلى من برنامجكم فاشل,negative,talbat
9,كتير في تحسن خدمة العملاء لطفين في بعض الاخطاء...,positive,talbat


In [3]:
arab_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40046 entries, 0 to 40045
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   review_description  40045 non-null  str  
 1   rating              40046 non-null  str  
 2   company             40046 non-null  str  
dtypes: str(3)
memory usage: 5.0 MB


In [4]:
arab_df["language"] = 'arabic'
arab_df.drop('company' , axis =1 ,inplace =True)
arab_df.head()


,review_description,rating,language
0,رائع,positive,arabic
1,برنامج رائع جدا يساعد على تلبيه الاحتياجات بشك...,positive,arabic
2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال با...,negative,arabic
3,لماذا لا يمكننا طلب من ماكدونالدز؟,negative,arabic
4,البرنامج بيظهر كل المطاعم و مغلقه مع انها بتكو...,negative,arabic


In [5]:
arab_df.rename(columns={'review_description' : 'review'} , inplace=True)
arab_df.rename(columns={'rating' : 'rating/sentiment'} , inplace=True)
arab_df.head()

,review,rating/sentiment,language
0,رائع,positive,arabic
1,برنامج رائع جدا يساعد على تلبيه الاحتياجات بشك...,positive,arabic
2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال با...,negative,arabic
3,لماذا لا يمكننا طلب من ماكدونالدز؟,negative,arabic
4,البرنامج بيظهر كل المطاعم و مغلقه مع انها بتكو...,negative,arabic


In [6]:
df2 = pd.read_csv("../Datasets/MovieReviewTrainingDatabase.csv")
eng_df = df2.copy()
eng_df.head(10)

,sentiment,review
0,Positive,With all this stuff going down at the moment w...
1,Positive,'The Classic War of the Worlds' by Timothy Hin...
2,Negative,The film starts with a manager (Nicholas Bell)...
3,Negative,It must be assumed that those who praised this...
4,Positive,Superbly trashy and wondrously unpretentious 8...
5,Positive,I dont know why people think this is such a ba...
6,Negative,"This movie could have been very good, but come..."
7,Negative,I watched this video at a friend's house. I'm ...
8,Negative,"A friend of mine bought this film for £1, and ..."
9,Positive,This movie is full of references. Like 'Mad ...


In [7]:
eng_df['language'] = 'English'
eng_df.rename(columns={'sentiment' : 'rating/sentiment'} , inplace=True)
eng_df.head()

,rating/sentiment,review,language
0,Positive,With all this stuff going down at the moment w...,English
1,Positive,'The Classic War of the Worlds' by Timothy Hin...,English
2,Negative,The film starts with a manager (Nicholas Bell)...,English
3,Negative,It must be assumed that those who praised this...,English
4,Positive,Superbly trashy and wondrously unpretentious 8...,English


In [8]:
df = pd.concat([arab_df , eng_df] , axis= 0)
df.head(-5)

,review,rating/sentiment,language
0,رائع,positive,arabic
1,برنامج رائع جدا يساعد على تلبيه الاحتياجات بشك...,positive,arabic
2,التطبيق لا يغتح دائما بيعطيني لا يوجد اتصال با...,negative,arabic
3,لماذا لا يمكننا طلب من ماكدونالدز؟,negative,arabic
4,البرنامج بيظهر كل المطاعم و مغلقه مع انها بتكو...,negative,arabic
...,...,...,...
24990,I've never been huge on IMAX films. They're co...,Positive,English
24991,Steve McQueen has certainly a lot of loyal fan...,Negative,English
24992,Sometimes you wonder how some people get fundi...,Negative,English
24993,"I am a student of film, and have been for seve...",Negative,English


In [9]:
df.describe()

,review,rating/sentiment,language
count,65045,65046,65046
unique,63907,5,2
top,ممتاز,positive,arabic
freq,16,23921,40046


In [10]:
df.info()

<class 'pandas.DataFrame'>
Index: 65046 entries, 0 to 24999
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   review            65045 non-null  str  
 1   rating/sentiment  65046 non-null  str  
 2   language          65046 non-null  str  
dtypes: str(3)
memory usage: 37.5 MB


In [11]:
df_full= df.sample(frac=1, random_state=42).reset_index(drop=True)
df_full.head()

,review,rating/sentiment,language
0,This two-part TV mini-series isn't as good as ...,Positive,English
1,انصفو الدرايفريه,negative,arabic
2,تضامن مع شبال طلبات الاردن,negative,arabic
3,One of the best films I've ever seen. Robert...,Positive,English
4,حبيت التطبيق سهل وسريع,positive,arabic


In [12]:
df_full[df_full['review'].duplicated(keep=False)].sort_values('review')

,review,rating/sentiment,language
51503,"Back in his youth, the old man had wanted to...",Negative,English
62423,"Back in his youth, the old man had wanted to...",Negative,English
52200,'Dead Letter Office' is a low-budget film abou...,Negative,English
54762,'Dead Letter Office' is a low-budget film abou...,Negative,English
45382,".......Playing Kaddiddlehopper, Col San Fernan...",Positive,English
...,...,...,...
27272,😡😡😡,negative,arabic
38607,🙂,negative,arabic
54544,🙂,negative,arabic
22461,🤗,positive,arabic


In [13]:
df_full['review'].duplicated().sum()

1138

In [14]:
df_full.groupby('review')['language'].nunique().value_counts()

language
1    63907
Name: count, dtype: int64

In [15]:
df_final = df_full.drop_duplicates()

In [16]:
df_final.shape

(64009, 3)

In [17]:
df_final['language'].value_counts()

language
arabic     39105
English    24904
Name: count, dtype: int64

In [18]:
df_final['review'].str.len().describe()

count    64008.000000
mean       539.845472
std        870.045893
min          1.000000
25%         23.000000
50%         79.000000
75%        788.000000
max      13603.000000
Name: review, dtype: float64

In [19]:
df_final[df_final['language'] == 'arabic']['review'].sample(10, random_state=42)

48233                                 ولاقلطةيستحق التنزيل
53438                              سريع وفعال للطلب السريع
64423    الباصات ممتازه وخدمه السائقين جيده ولاكن الابل...
14476                        اريد الكثير من المطاعم الصحيه
23260      البرنامج اكثر من رائع وحتى طلبك من المطاعم ممتع
14675                                 حلو و احسن مما نتكلم
4116     لا انصح بتحميل التطبيق حصل معي موقف عند توصيل ...
13788                                              فنتستيك
46300                                            غالين مره
21922                   ممتاز ومحترم في التعامل مع الجمهور
Name: review, dtype: str

In [20]:
df_final[df_final['language'] == 'English']['review'].sample(10, random_state=42)

32846    Like most people I was intrigued when I heard ...
7867     I watched the trailer on the DVD after seeing ...
62520    Audiard made here a very interesting movie. It...
11377    This movie surprised me. Some things were 'cli...
11869    Burt Reynolds came to a point in his career wh...
32224    79/100. Fred Astaire and Ginger Rogers never m...
10251    I cherish each and every frame of this beautif...
50330    This has to be one of the worst films of the 1...
9743     Being a fan of Billy Bob Thornton, and the div...
15533    The literary genius of Vladimir Navokov is bro...
Name: review, dtype: str

In [21]:
df_final['review'][0]

"This two-part TV mini-series isn't as good as the original from 1966 but it's solid. The original benefited from a huge number of things---it was all in black and white, it had a great jazz score and it was filmed at the real locations, including the home of the doomed Clutter family. That was important because in the book and in the original movie the home is very much a character itself.  This remake was filmed in Canada which I guess doubles okay for Kansas. The story tries to be as sympathetic to Perry as it dares to and Eric Roberts plays him as a somewhat fey person, his homosexuality barely hidden. The gentler take by Roberts doesn't quite work in the end though because it's hard to believe that his version of Perry Smith would just finally explode in a spasm of murder. Whereas Robert Blake's take on Smith left you no doubt that his Perry Smith was an extremely dangerous character.  Anthony Edwards was excellent as the bombastic, big-mouthed and ultimately cowardly Dick Hickcoc

In [22]:
df_final[df_final['language'] == 'arabic']['review'].str.len().describe()

count    39104.000000
mean        51.827997
std         68.742753
min          1.000000
25%         15.000000
50%         29.000000
75%         62.000000
max       2481.000000
Name: review, dtype: float64

In [23]:
df_final[df_final['language'] == 'English']['review'].min()

"\x08\x08\x08\x08A Turkish Bath sequence in a film noir located in New York in the 50's, that must be a hint at something ! Something that curiously, in all the previous comments, no one has pointed out , but seems to me essential to the understanding of this movie   the Turkish Baths sequence: a back street at night, the entrance of a sleazy sauna, and Scalise wrapped in a sheet, getting his thighs massaged. Steve, the masseur is of the young rough boxer ( Beefcake!) type , and another guy, a bodyguard? finishes dressing up. Dixon obviously hates what he sees there and gets rough right away. We know he has a reputation for roughing up suspects. Good cop but getting out of control easy. Why is it that he hates them so much ?   Could it be that he hates himself. This part of himself he inherited from his father ? That dark side that could lead him right at the end of the sidewalk, into the gutter ? What if that dark side lurked within a 'closet' ? Remember : whenever Dixon meets Scalise

In [24]:
df_final[df_final['language'] == 'arabic'].assign(
    length=lambda x: x['review'].str.len()
).sort_values('length')[['review', 'length']].head(50)

,review,length
37253,💞,1.0
48064,🤐,1.0
4424,🤣,1.0
18147,🙊,1.0
39257,👆,1.0
48160,💖,1.0
693,😉,1.0
48431,🎃,1.0
10571,😇,1.0
5747,😑,1.0


In [25]:
df_final[df_final['language'] == 'English'].assign(
    length=lambda x: x['review'].str.len()
).sort_values('length')[['review', 'length']].head(20)

,review,length
42746,This movie is terrible but it has some good ef...,52
12372,I wouldn't rent this one even on dollar rental...,53
47389,Ming The Merciless does a little Bardwork and ...,64
3634,You'd better choose Paul Verhoeven's even if y...,65
4208,Adrian Pasdar is excellent is this film. He ma...,70
2177,"Long, boring, blasphemous. Never have I been s...",80
12178,"I don't know why I like this movie so well, bu...",81
43773,"no comment - stupid movie, acting average or w...",94
23952,A rating of '1' does not begin to express how ...,99
7928,This is the definitive movie version of Hamlet...,102


In [26]:
arabic_pattern = r'[؀-ۿ]'
english_pattern = r'[A-Za-z]'

non_linguistic = df_final[
    ~(
        df_final['review'].astype(str).str.contains(arabic_pattern, regex=True, na=False)
        |
        df_final['review'].astype(str).str.contains(english_pattern, regex=True, na=False)
    )
]

In [27]:
len(non_linguistic)

781

In [28]:
non_linguistic[['review', 'language']].head(50)

,review,language
74,:) 👍,arabic
158,🥰😘🍒👍,arabic
299,😕,arabic
384,👍😎,arabic
473,👍🏻,arabic
482,:*,arabic
543,👎👎,arabic
650,😄👍,arabic
693,😉,arabic
699,😁😁😁😁😁😁😁😁😁,arabic


In [29]:
non_linguistic['language'].value_counts()

language
arabic    781
Name: count, dtype: int64

In [30]:
df_final = df_final.drop(index=non_linguistic.index)

In [31]:
arabic_pattern = r'[؀-ۿ]'
english_pattern = r'[A-Za-z]'

non_linguistic = df_final[
    ~(
        df_final['review'].astype(str).str.contains(arabic_pattern, regex=True, na=False)
        |
        df_final['review'].astype(str).str.contains(english_pattern, regex=True, na=False)
    )
]

len(non_linguistic)

0

In [32]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+")
MENTION_HASHTAG_RE = re.compile(r"[@#]\w+")
HTML_RE = re.compile(r"<.*?>")
NUMBER_RE = re.compile(r"\d+")
MULTISPACE_RE = re.compile(r"\s+")

# Arabic diacritics (tashkeel) + tatweel/kashida
ARABIC_DIACRITICS_RE = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0670\u0640]")

# Keep Arabic letters, Latin letters, and basic whitespace; drop punctuation/symbols
PUNCT_TABLE = str.maketrans("", "", string.punctuation + "،؛؟«»ـ")


def clean_text(text: str) -> str:
    text = str(text)
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)
    text = MENTION_HASHTAG_RE.sub(" ", text)
    text = HTML_RE.sub(" ", text)
    text = NUMBER_RE.sub(" ", text)

    # Arabic-specific normalization
    text = ARABIC_DIACRITICS_RE.sub("", text)

    # Lowercase (only affects English half; Arabic has no case)
    text = text.lower()

    # Strip punctuation
    text = text.translate(PUNCT_TABLE)

    # Collapse whitespace
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text


df_final["clean_review"] = df_final["review"].apply(clean_text)

# Drop rows that became empty after cleaning
df_final = df_final[df_final["clean_review"].str.len() > 0].reset_index(drop=True)

df_final[["review", "clean_review", "language"]].head(50)

,review,clean_review,language
0,This two-part TV mini-series isn't as good as ...,this twopart tv miniseries isnt as good as the...,English
1,انصفو الدرايفريه,انصفو الدرايفريه,arabic
2,تضامن مع شبال طلبات الاردن,تضامن مع شبال طلبات الاردن,arabic
3,One of the best films I've ever seen. Robert...,one of the best films ive ever seen robert duv...,English
4,حبيت التطبيق سهل وسريع,حبيت التطبيق سهل وسريع,arabic
5,ياخي يفك من ازمات واذا ما عندك رصيد يفزعلك بس ...,ياخي يفك من ازمات واذا ما عندك رصيد يفزعلك بس ...,arabic
6,What a disappointment! I hated the mummy but t...,what a disappointment i hated the mummy but th...,English
7,I just finished reading a book about Dillinger...,i just finished reading a book about dillinger...,English
8,روعه . خطير مره,روعه خطير مره,arabic
9,This film is not deserved of the next few minu...,this film is not deserved of the next few minu...,English


In [33]:
X = df_final["clean_review"]
y = df_final["language"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [34]:
vectorizer = TfidfVectorizer(
    analyzer="char_wb",   # char n-grams within word boundaries
    ngram_range=(2, 4),
    min_df=2,
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [35]:
models = {
    "MultinomialNB": MultinomialNB(),
    "LinearSVC": LinearSVC(random_state=42),
    "LogisticRegression": LogisticRegression( max_iter=1000, random_state=42 ),
}

results = []
fitted_models = {}

for name, model in models.items():
    start_train = time.time()
    model.fit(X_train_vec, y_train)
    train_time = time.time() - start_train

    start_pred = time.time()
    val_preds = model.predict(X_test_vec)
    predict_time = time.time() - start_pred

    acc = accuracy_score(y_test, val_preds)
  

    fitted_models[name] = model
    results.append({
        "model": name,
        "val_accuracy": acc,
        "train_time_sec": train_time,
        "predict_time_sec": predict_time,
    })

result = pd.DataFrame(results).sort_values("val_accuracy", ascending=False)
result

,model,val_accuracy,train_time_sec,predict_time_sec
1,LinearSVC,0.999525,1.022182,0.023999
2,LogisticRegression,0.998812,1.817876,0.020997
0,MultinomialNB,0.984875,0.593665,0.062997


In [36]:
best_model_name = result.iloc[0]["model"]

In [37]:
best_model = fitted_models[best_model_name]
joblib.dump(best_model, 'Language_classifier_weights.pkl')
joblib.dump(vectorizer, 'Language_classifier_Vectorizer.pkl')
print("Best model saved to Language_classifier_weights.pkl")

Best model saved to Language_classifier_weights.pkl
